# VTX-300M Cloud Training Notebook

Standalone notebook for a fresh cloud GPU runtime. It can clone this repo, install dependencies, log in to Hugging Face, run pretraining with automatic checkpoint resume, optionally evaluate/distill, and push checkpoints or final artifacts to the Hub.

Before running the training cells, set `HF_TOKEN` and update the repo IDs in the config cell.

## 1. Runtime Setup

In [ ]:
import os, sys, json, subprocess, textwrap, pathlib, shutil
from pathlib import Path

def run(cmd, check=True, cwd=None, env=None):
    print(f"\n$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)

print("Python:", sys.version)
run("nvidia-smi", check=False)

## 2. Get The Repo

If you already uploaded the repo into the notebook runtime, set `REPO_DIR` to that folder and set `CLONE_REPO = False`.

In [ ]:
CLONE_REPO = True
GIT_REPO_URL = "https://github.com/OEvortex/VTX-300M.git"
REPO_DIR = Path("/content/VTX-300M")

if CLONE_REPO:
    if REPO_DIR.exists():
        print(f"Repo already exists: {REPO_DIR}")
    else:
        run(f"git clone {GIT_REPO_URL} {REPO_DIR}")
else:
    REPO_DIR = Path.cwd()

SRC_DIR = REPO_DIR / "src"
assert (SRC_DIR / "pretrain.py").exists(), f"Could not find src/pretrain.py under {REPO_DIR}"
print("Repo:", REPO_DIR)
print("Source:", SRC_DIR)

## 3. Install Dependencies

The PyTorch command below targets CUDA 12.4. If your cloud image already has working PyTorch, you can set `INSTALL_TORCH = False`.

In [ ]:
INSTALL_TORCH = True
INSTALL_DISTILLKIT = False  # Set True only if you plan to run distillation.

run(f"{sys.executable} -m pip install -U pip setuptools wheel")

if INSTALL_TORCH:
    run(f"{sys.executable} -m pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")

base_deps = [
    "transformers>=4.45.0",
    "huggingface_hub>=0.24.0",
    "hf_transfer",
    "datasets",
    "accelerate",
    "safetensors",
    "numpy",
    "scipy",
    "trackio",
    "fastapi",
    "uvicorn",
    "pydantic",
]
run(f"{sys.executable} -m pip install -U " + " ".join(base_deps))

if INSTALL_DISTILLKIT:
    run(f"{sys.executable} -m pip install -U distill-kit")

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 4. Hugging Face Login

Use a token with write access for pushing models. In Colab, you can paste it when prompted or set it as a secret/env var named `HF_TOKEN`.

In [ ]:
from getpass import getpass
from huggingface_hub import login, whoami, HfApi, create_repo

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

login(token=HF_TOKEN, add_to_git_credential=False)
print(whoami(token=HF_TOKEN))

## 5. Configure Training

For a smoke test, set `STEPS = 5` and `PUSH_EVERY = 5`. For real training, increase `STEPS` and adjust batch settings for your GPU memory.

In [ ]:
MODEL_REPO_ID = "VTXAI/vtx-300m"          # Change to your HF namespace/repo.
DATA_SHARDS = "AUTO"                      # AUTO downloads default Hub data, or use a list of local .bin paths.
SAVE_DIR = "/content/vtx_300m_ckpt"       # Persistent mounted drive path is better for long runs.

STEPS = 76000
WARMUP = 1000
LR = 6e-4
MIN_LR = 6e-5
BATCH = 4
GRAD_ACCUM = 8
BLOCK = 2048
PUSH_EVERY = 3000
LOG_EVERY = 25
COMPILE = True

TRACKIO_SPACE_ID = "VTXAI/vtx-300m-trackio"  # Set None to disable Trackio.
TRACKIO_PROJECT = "vtx-300m"

train_config = {
    "steps": STEPS,
    "warmup": WARMUP,
    "lr": LR,
    "min_lr": MIN_LR,
    "weight_decay": 0.1,
    "beta1": 0.9,
    "beta2": 0.95,
    "grad_clip": 1.0,
    "batch": BATCH,
    "grad_accum": GRAD_ACCUM,
    "block": BLOCK,
    "seed": 42,
    "shards": DATA_SHARDS,
    "hub_repo": MODEL_REPO_ID,
    "trackio_space": TRACKIO_SPACE_ID,
    "trackio_project": TRACKIO_PROJECT,
    "push_every": PUSH_EVERY,
    "token": HF_TOKEN,
    "log_every": LOG_EVERY,
    "save_dir": SAVE_DIR,
    "compile": COMPILE,
}

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
CONFIG_PATH = SRC_DIR / "cloud_pretrain_config.json"
CONFIG_PATH.write_text(json.dumps(train_config, indent=2))
print(CONFIG_PATH.read_text())

## 6. Create Hub Repos

In [ ]:
create_repo(MODEL_REPO_ID, token=HF_TOKEN, private=False, exist_ok=True)
if TRACKIO_SPACE_ID:
    create_repo(TRACKIO_SPACE_ID, token=HF_TOKEN, repo_type="space", space_sdk="static", private=False, exist_ok=True)
print("Hub repos ready")

## 7. Run Pretraining

This resumes automatically from the newest `step_*` checkpoint inside `SAVE_DIR`. To start fresh, add `--no-resume` to the command.

In [ ]:
env = os.environ.copy()
if TRACKIO_SPACE_ID:
    env["TRACKIO_SPACE_ID"] = TRACKIO_SPACE_ID
    env["TRACKIO_PROJECT"] = TRACKIO_PROJECT

cmd = f"{sys.executable} pretrain.py --config {CONFIG_PATH.name}"
run(cmd, cwd=SRC_DIR, env=env)

## 8. Find Latest Local Checkpoint

In [ ]:
def latest_checkpoint(save_dir):
    root = Path(save_dir)
    steps = []
    for p in root.glob("step_*"):
        if p.is_dir():
            try:
                steps.append((int(p.name.split("_", 1)[1]), p))
            except Exception:
                pass
    if not steps:
        return None
    return sorted(steps)[-1][1]

LATEST_CKPT = latest_checkpoint(SAVE_DIR)
print("Latest checkpoint:", LATEST_CKPT)

## 9. Push Latest Checkpoint Manually

Training already pushes every `PUSH_EVERY` steps and at the final step when `MODEL_REPO_ID` is set. Use this cell if you stopped early or want to push the latest local checkpoint.

In [ ]:
PUSH_LATEST_MANUALLY = False

if PUSH_LATEST_MANUALLY:
    assert LATEST_CKPT is not None, "No checkpoint found"
    api = HfApi(token=HF_TOKEN)
    api.upload_folder(
        folder_path=str(LATEST_CKPT),
        repo_id=MODEL_REPO_ID,
        token=HF_TOKEN,
        commit_message=f"manual upload {LATEST_CKPT.name}",
    )
    print(f"Pushed {LATEST_CKPT} to {MODEL_REPO_ID}")

## 10. Optional Evaluation

Set `RUN_EVAL = True` after you have a checkpoint or Hub model to evaluate. Use `EVAL_LIMIT` for a quick smoke test.

In [ ]:
RUN_EVAL = False
EVAL_CKPT = str(LATEST_CKPT or MODEL_REPO_ID)
EVAL_LIMIT = 50  # Set None for full eval.
EVAL_OUT = str(REPO_DIR / "vtx_300m_eval.json")

if RUN_EVAL:
    limit_arg = "" if EVAL_LIMIT is None else f" --limit {EVAL_LIMIT}"
    run(
        f"{sys.executable} eval_benchmarks.py --ckpt {EVAL_CKPT} --tokenizer Qwen/Qwen3-4B --batch 8 --out {EVAL_OUT}{limit_arg}",
        cwd=SRC_DIR,
        env=os.environ.copy(),
    )
    print(Path(EVAL_OUT).read_text())

## 11. Optional Distillation

This requires `INSTALL_DISTILLKIT = True` in the install cell. Distillation can be expensive; run only after pretraining has produced a usable student checkpoint.

In [ ]:
RUN_DISTILL = False
DISTILLED_REPO_ID = "VTXAI/vtx-300m-distilled"  # Change to your namespace/repo.
DISTILL_STUDENT = MODEL_REPO_ID
DISTILL_STEPS = 8000
DISTILL_LR = 2e-5
DISTILL_BATCH = 2
DISTILL_GRAD_ACCUM = 8
DISTILL_MAX_SEQ_LEN = 2048

if RUN_DISTILL:
    create_repo(DISTILLED_REPO_ID, token=HF_TOKEN, private=False, exist_ok=True)
    run(
        " ".join([
            f"{sys.executable} distill.py",
            f"--student-ckpt {DISTILL_STUDENT}",
            "--distill-config distill_config.yml",
            f"--out {DISTILLED_REPO_ID}",
            f"--steps {DISTILL_STEPS}",
            f"--lr {DISTILL_LR}",
            f"--batch {DISTILL_BATCH}",
            f"--grad-accum {DISTILL_GRAD_ACCUM}",
            f"--max-seq-len {DISTILL_MAX_SEQ_LEN}",
        ]),
        cwd=SRC_DIR,
        env=os.environ.copy(),
    )

## 12. Download / Backup Artifacts

If your cloud runtime is temporary, push to the Hub or mount persistent storage before long training. This cell prints the local checkpoint tree.

In [ ]:
run(f"find {SAVE_DIR} -maxdepth 2 -type f | sed -n '1,120p'", check=False)
print("Model repo:", f"https://huggingface.co/{MODEL_REPO_ID}")
if TRACKIO_SPACE_ID:
    print("Trackio space:", f"https://huggingface.co/spaces/{TRACKIO_SPACE_ID}")